In [18]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from utils import extract_url_map
from events import extract_events
from timespan import parse_timespan
from taxonomy import load_taxonomy

In [19]:
root_path = "../data/schede mappatura/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
}

In [20]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede = [c for c in chrono_schede if not c.endswith(":Zone.Identifier")]
chrono_schede

['../data/schede mappatura/0_template/chronotopoi_template.xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx',
 '../data/schede mappatura/bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx',
 '../data/schede mappatura/david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx']

## A list of individual sources for experimentation

ignored in the oveall logic

In [21]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

['../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx']

After identification of all sources

# Shortlist processable sources

In [22]:
overview = {
    "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [23]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = [
            "event_label",
            "event_type",
            "place_name",
            "place_type",
            "place_category",
            "wikidata_qid",
            "geonames_id",
            "google maps",
            "date_certainty",
            "date_label",
            "memorial_inscription",
            "source_doc",
            "source_timecode",
            "source_quote",
            "external_links",
            "notes",
        ]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        # df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)

# Track start/end locations: end_location = current row's place,
# start_location = previous event's place (per person)
start_locations = []
prev_location = {}  # protagonist → last place_name
for _, row in df.iterrows():
    person = row["protagonist"]
    end_loc = row["place_name"].strip()
    start_loc = prev_location.get(person, end_loc)  # fallback to same as end
    start_locations.append(start_loc)
    if end_loc:
        prev_location[person] = end_loc
df["start_location"] = start_locations
df["end_location"] = df["place_name"]

# Collect concepts from event_type, place_type, place_category
concept_labels = set()
for col in ["event_type", "place_type", "place_category"]:
    for val in df[col]:
        v = val.strip()
        if v and v not in ("nan", "None"):
            concept_labels.add(v)
print(f"Concepts to create: {sorted(concept_labels)}")

df


stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx Josef Stern Index(['event_label', 'event_type', 'place_name', 'place_type',
       'place_category', 'wikidata_qid', 'geonames_id', 'google maps',
       'date_certainty', 'date_label', 'memorial_inscription', 'source_doc',
       'source_timecode', 'source_quote', 'external_links', 'notes'],
      dtype='str')
Concepts to create: ['Bahnhof', 'Ghettohaus', 'Schiff', 'Synagoge', 'Telephon im Ghettohaus', 'Zug', 'alte_heimat', 'birth', 'education', 'emigration', 'emigration_transit', 'farewell', 'forced_eviction', 'immigration', 'marriage', 'military_service', 'neue_heimat', 'notification', 'ort_der_zeit', 'reise_zurueck', 'residence', 'return_visit', 'return_visit_transit', 'verkehrsmittel', 'work']


,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,memorial_inscription,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name,start_location,end_location
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11","Gießen, Marktplatz 11"
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11",Gießen
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,Gießen,Gießen
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,Gießen,Löberstraße 20
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,Löberstraße 20,Gießen
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,Gießen,"Ghettohaus, Walltorstraße 48"
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,"Ghettohaus, Walltorstraße 48",Schlesien
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,Schlesien,Berlin
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,Berlin,"Synagoge am Börneplatz, Frankfurt/M"
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"Synagoge am Börneplatz, Frankfurt/M","Bahnhof, Frankfurt am Main"


# Locations

In [24]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip().removesuffix(".0")
        )

    locs[name].update(urls[0])
print(locs)

100%|██████████| 33/33 [00:00<00:00, 49521.30it/s]

{'Gießen, Marktplatz 11': {'label': 'ort_der_zeit', 'www.geonames.org': 'https://www.geonames.org/2920512'}, 'Gießen': {'label': 'reise_zurueck', 'www.geonames.org': 'https://www.geonames.org/2920512/giessen.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q564579'}, 'Löberstraße 20': {'label': 'alte_heimat', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Ghettohaus, Walltorstraße 48': {'label': 'alte_heimat,Ghettohaus', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Schlesien': {'label': 'alte_heimat'}, 'Berlin': {'label': 'alte_heimat', 'www.geonames.org': 'https://www.geonames.org/2950159/berlin.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q64'}, 'Synagoge am Börneplatz, Frankfurt/M': {'label': 'alte_heimat,Synagoge', 'www.geonames.org': 'https://www.geonames.org/6553153/frankfurt-am-main.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q17

## Add GO concepts as locations

Concepts under 'GO – Luoghi geografici' are geographic locations.
Add them to the locs dict so they get included in locations.xlsx.

In [25]:
from taxonomy import load_taxonomy

taxonomy = load_taxonomy()

go_concepts = [
    label for label, cat in taxonomy.concept_to_category.items()
    if cat == "GO"
]
# Also include GO sub-category labels
for key, info in taxonomy.sub_categories.items():
    if info.get("parent") == "GO":
        go_concepts.append(info["label"])

for concept_name in go_concepts:
    concept_name = concept_name.strip()
    if not concept_name:
        continue
    if concept_name not in locs:
        locs[concept_name] = {}
    if "label" not in locs[concept_name] or not locs[concept_name]["label"]:
        locs[concept_name]["label"] = "GO"

print(f"Added {len(go_concepts)} GO concepts to locations, total: {len(locs)}")


Added 98 GO concepts to locations, total: 119


## Update preexisting locations

In [26]:
import os
import re
from locations import enrich_locations_xlsx

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    rich = pd.read_excel("locations.xlsx", dtype=str)
    rich = rich.set_index(["location"])
else:
    rich = pd.DataFrame()
    rich.index.name = "location"

# Build a normalized index for flexible matching
existing_keys = {_normalize_loc(idx): idx for idx in rich.index}

for location, row in locs.items():
    key = _normalize_loc(location)

    if key in existing_keys:
        # Enrich existing row: only fill absent cells
        real_idx = existing_keys[key]
        if isinstance(row, dict):
            for col, val in row.items():
                if col not in rich.columns:
                    rich[col] = ""
                existing = rich.loc[real_idx, col]
                if isinstance(existing, pd.Series):
                    existing = existing.iloc[0]
                if pd.isna(existing) or str(existing).strip() in ("", "nan"):
                    rich.loc[real_idx, col] = str(val)
    else:
        # New location: add row with provided values
        if isinstance(row, dict):
            for col in row:
                if col not in rich.columns:
                    rich[col] = ""
            rich.loc[location] = {col: str(val) for col, val in row.items()}
        else:
            rich.loc[location] = pd.Series(dtype=str)
        existing_keys[key] = location

rich.to_excel("locations.xlsx")

# Enrich with bag-of-words and super-region columns
enrich_locations_xlsx("locations.xlsx")

In [30]:
taxonomy = load_taxonomy("../data/maxqda/MAXQDA_Code_System.mmd")
events = sorted(set(taxonomy.concept_to_category.keys()), key=lambda x: -len(x))
len(events), events[:15] + ["..."] + events[-15:]
# ",".join(events)

(326,
 ['Versteck bei Familie / Privatperson',
  'IPD (Jüd. Pfadfinder Deutschland)',
  'Kloster / kirchliche Einrichtung',
  'Hilfsverein der deutschen Juden',
  'Palästinaamt (Meinekestraße 10)',
  'Friedrichswerdersche Gymnasium',
  'Deutsche Diakonissenhospital',
  'Staatsgründung Israel (1948)',
  'Soldat der Jüdischen Brigade',
  'Gymnasium zum Grauen Kloster',
  'Novemberrevolution (1918–19)',
  'Reflexion in Erzählgegenwart',
  'Soldat der britischen Armee',
  'Blindenwerkstatt Otto Weidt',
  'Synagoge Gießen (orthodox)',
  '...',
  '1934',
  'Prag',
  'Kehl',
  '1921',
  '1986',
  '1939',
  '1919',
  'Emek',
  '1915',
  '1928',
  'SPD',
  'Zug',
  'KPD',
  'Tod',
  'USA'])

# Timespan

In [27]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,...,source_timecode,source_quote,external_links,notes,protagonist,name,start_location,end_location,time_start,time_end
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,...,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11","Gießen, Marktplatz 11",1921-06-15,1921-06-15
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,...,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11",Gießen,1928-01-01,1932-12-31
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,...,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,Gießen,Gießen,1932-01-01,1932-12-31
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,...,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,Gießen,Löberstraße 20,1933-01-01,1933-12-31
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,...,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,Löberstraße 20,Gießen,1935-01-01,1935-12-31
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,...,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,Gießen,"Ghettohaus, Walltorstraße 48",1935-01-01,1935-12-31
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,...,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,"Ghettohaus, Walltorstraße 48",Schlesien,1935-01-01,1935-12-31
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,...,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,Schlesien,Berlin,1935-01-01,1935-12-31
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,...,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,Berlin,"Synagoge am Börneplatz, Frankfurt/M",1936-01-01,1936-12-31
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,...,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"Synagoge am Börneplatz, Frankfurt/M","Bahnhof, Frankfurt am Main",1936-01-01,1936-12-31


# Notes

left unprocessed for now

In [28]:
set(df["notes"])

{'11974166.0 https://www.wikidata.org/wiki/Q116016915\nhttps://www.geonames.org/11974166/grossen-linden.html',
 '2888549.0 https://www.wikidata.org/wiki/Q1571834\nhttps://www.geonames.org/2888549/klein-linden.html',
 '2891951.0 probable 1936 https://www.wikidata.org/wiki/Q15979\nhttps://www.geonames.org/2891951/kehl.html',
 '2920512.0 certain 15/06/1921 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579\nhttps://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 '2920512.0 probable 1975 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579',
 '293304.0 uncertain 1940-1944 https://www.wikidata.org/wiki/Q550025\nhttps://www.geonames.org/293304/tirat-tsvi.html',
 '2950159.0 probable 1935 31min 31s https://www.wikidata.org/wiki/Q64\nhttps://www.geonames.org/2950159/berlin.html',
 '2995469.0 probable 1936 https://www.wikidata.org/wiki/Q23482\nhttps://www.geonames.org/2995469/marseille.html',
 '31min 58

# Links

left unprocessed for now

In [29]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

{'https://www.geonames.org/11974166/grossen-linden.html',
 'https://www.geonames.org/2888549/klein-linden.html',
 'https://www.geonames.org/2891951/kehl.html',
 'https://www.geonames.org/2920512/giessen.html',
 'https://www.geonames.org/293165/jezreel-valley.html',
 'https://www.geonames.org/293304/tirat-tsvi.html',
 'https://www.geonames.org/294801/haifa.html',
 'https://www.geonames.org/2950159/berlin.html',
 'https://www.geonames.org/295211/-en-hanaziv.html',
 'https://www.geonames.org/2995469/marseille.html',
 'https://www.geonames.org/6290300/frankfurt-hauptbahnhof.html',
 'https://www.geonames.org/6553153/frankfurt-am-main.html',
 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 'https://www.wikidata.org/wiki/Q111620976',
 'https://www.wikidata.org/wiki/Q116016915',
 'https://www.wikidata.org/wiki/Q1375288',
 'https://www.wikidata.org/wiki/Q1571834',
 'https://www.wikidata.org/wiki/Q15979',
 'https://www.wikidata.org/wiki/Q165368',
 'https://ww

# Events

In [31]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,...,source_quote,external_links,notes,protagonist,name,start_location,end_location,time_start,time_end,event
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,...,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11","Gießen, Marktplatz 11",1921-06-15,1921-06-15,"[Geburt, Alte Heimat]"
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,...,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"Gießen, Marktplatz 11",Gießen,1928-01-01,1932-12-31,[Alte Heimat Grundschule]
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,...,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,Gießen,Gießen,1932-01-01,1932-12-31,"[Realgymnasium, Alte Heimat]"
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,...,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,Gießen,Löberstraße 20,1933-01-01,1933-12-31,[Wohnort]
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,...,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,Löberstraße 20,Gießen,1935-01-01,1935-12-31,[Abgang von der Schule]
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,...,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,Gießen,"Ghettohaus, Walltorstraße 48",1935-01-01,1935-12-31,[Wohnort]
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,...,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,"Ghettohaus, Walltorstraße 48",Schlesien,1935-01-01,1935-12-31,[Hachschara]
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,...,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,Schlesien,Berlin,1935-01-01,1935-12-31,"[Verwandte, Bei n]"
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,...,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,Berlin,"Synagoge am Börneplatz, Frankfurt/M",1936-01-01,1936-12-31,[Jeschiwa]
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,...,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"Synagoge am Börneplatz, Frankfurt/M","Bahnhof, Frankfurt am Main",1936-01-01,1936-12-31,"[Abschied von den Eltern, Abfahrt >]"


In [32]:
from api_client import (
    login, get_or_create_concept,
)
from locations import (
    load_locations_db,
)

login()
load_locations_db()

# === Main import ===

# 1. Create all concepts
print("Creating concepts...")
for label in sorted(concept_labels):
    get_or_create_concept(label)
print(f"  {len(concept_labels)} concept labels processed")

Authenticated
  Loaded 244 locations from locations.xlsx
Creating concepts...
  25 concept labels processed
